### **Data Reading**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format("parquet")\
        .load("abfss://bronzestg@storageloweretep1.dfs.core.windows.net/orders")


In [0]:
# TO check the schema
df.printSchema()

In [0]:
df=df.withColumnRenamed("_rescued_data", "rescued_data")

In [0]:
df = df.drop("rescued_data")

_Date Transformation_

In [0]:
df = df.withColumn("order_date",to_timestamp(col('order_date')))

In [0]:
df = df.withColumn("year", year(col("order_date")))

_denseRank_

In [0]:
df1 = df.withColumn("flag", dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))

_Rank_

In [0]:
df1 = df1.withColumn("rank_flag", rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))

_rowNumber_

In [0]:
df1 = df1.withColumn("row_number", row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))

### **Classes - OOPS**

In [0]:
class windows:
   
    def denseRank(self,df):
        df_dense_rank = df.withColumn("flag", dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_dense_rank
    
    def rank(self,df):
        df_rank = df.withColumn("rank", rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_rank
    
    def rowNumber(self,df):
        df_row_number = df.withColumn("row_number", row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_row_number

In [0]:
df_new = df
obj = windows()
df_denseRank=obj.denseRank(df_new)
df_rowNumber=obj.rowNumber(df_denseRank)
df_rank = obj.rank(df_rowNumber)

In [0]:
df_rank.display()

### _Data Writing_

In [0]:
df.write.format("delta").mode("append").save("abfss://silverstg@storageloweretep1.dfs.core.windows.net/orders")
        

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databrickscatalogetep1.silver.orderSilver
        USING DELTA 
        LOCATION 'abfss://silverstg@storageloweretep1.dfs.core.windows.net/orders'